# Notebook 4B — Generation Stage (Range 25%–50%)

Processes 25%–50% of all runs.
Run A, B, C, D in parallel — each writes to its own parquet file.

**Input:** `data/retail/retrieval/20260413_retrieval_results_geo_v2.csv`

**Output:** `data/retail/generation/4_generation_results_B_v3.parquet`

## Instructions
1. Run **Setup**
2. Run **Data Preparation**
3. Run **Helper Functions**
4. Run **Parameters** — range is set automatically
5. Run **Generation**
6. Run **Inspect Results**
7. Run **Summary**
8. After all 4 notebooks finish: run **Merge Results** in any one notebook

## Setup

In [1]:
# Library imports
import json
import os
import re
import sys
import time
from datetime import datetime

import pandas as pd
import numpy as np

# Go up two levels to project root
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, "..", ".."))

# Add src/ to path so OpenAIHelper can be imported
sys.path.insert(0, os.path.join(project_root, "src"))


from llms import OpenAIHelper

# Load API key from config.json
config_path = os.path.join(project_root, "config.json")
with open(config_path, "r") as f:
    config = json.load(f)
os.environ["OPENAI_API_KEY"] = config["OPENAI_API_KEY"]

data_dir = os.path.join(project_root, "data", "retail")

print("Setup complete.")
print(f"Project root: {project_root}")

Setup complete.
Project root: /Users/leonardrampf/Library/CloudStorage/OneDrive-Personal/Dokumente/Universität/Nova SBE/Work Project/geo-experiment


## Data Preparation

Load the retrieval CSV and extract `original_query_id` and `method` from `query_id`.
Example: `50881_Fluency(doc)` → `original_query_id=50881`, `method=Fluency(doc)`

In [2]:
# Load the retrieval CSV — one row per document per query_id
# query_id format: "{original_query_id}_{method}"
retrieval_csv_path = os.path.join(data_dir, "retrieval/20260413_retrieval_results_geo_v2.csv")
df = pd.read_csv(retrieval_csv_path)

# Extract original_query_id (numeric) and method name from compound query_id
df["original_query_id"] = df["query_id"].str.extract(r"^(\d+)_")
df["method"]            = df["query_id"].str.extract(r"^\d+_(.+)$")

# Identify runs where the target document is missing from the retrieval results
# These runs are skipped during generation (no valid citation position can be computed)
target_per_run = df.groupby("query_id")["is_target"].sum()
runs_with_target    = set(target_per_run[target_per_run == 1].index)
runs_without_target = set(target_per_run[target_per_run == 0].index)

# Save runs without target for documentation and reproducibility
runs_without_target_path = os.path.join(data_dir, "generation/4_generation_runs_without_target_v3.json")
with open(runs_without_target_path, "w") as f:
    json.dump(sorted(list(runs_without_target)), f, indent=4)

print(f"Total rows:              {len(df)}")
print(f"Unique runs (query_ids): {df['query_id'].nunique()}")
print(f"Unique original queries: {df['original_query_id'].nunique()}")
print(f"Unique methods:          {df['method'].nunique()}")
print(f"Runs with target:        {len(runs_with_target)}")
print(f"Runs without target:     {len(runs_without_target)} — saved to generation/4_generation_runs_without_target_v3.json")
print()
print("Methods found:")
for m in sorted(df['method'].unique()):
    print(f"  {m}")

Total rows:              109875
Unique runs (query_ids): 11000
Unique original queries: 500
Unique methods:          22
Runs with target:        10985
Runs without target:     15 — saved to generation/4_generation_runs_without_target_v3.json

Methods found:
  Authoritative(doc)
  CQ(doc)
  CQS(doc)
  CS(doc)
  Citations(doc)
  ContentImprovement(doc)
  FC(doc)
  FCQ(doc)
  FCQS(doc)
  FCS(doc)
  FQ(doc)
  FQS(doc)
  FS(doc)
  Fluency(doc)
  LLMstxt(doc)
  QS(doc)
  Quotes(doc)
  SimpleLanguage(doc)
  Statistics(doc)
  TechnicalTerms(doc)
  UniqueWords(doc)
  doc


## Helper Functions

In [3]:
# System prompt for the LLM — instructs it to act as a product recommender and to cite every sentence with inline references [1], [2], etc.
DEVELOPER_PROMPT = """Your are a product recommender system for an online marketplace. Write an accurate and concise answer for the given user question, using _only_ the provided summarized web search results. The answer should be correct, high-quality, and written by an expert using an unbiased and journalistic tone. The answer should be informative, interesting, and engaging. The answer's logic and reasoning should be rigorous and defensible. Every sentence in the answer should be _immediately followed_ by an in-line citation to the search result(s). The cited search result(s) should fully support _all_ the information in the sentence. Search results need to be cited using [index]. When citing several search results, use [1][2][3] format rather than [1, 2, 3]. You can use multiple search results to respond comprehensively while avoiding irrelevant search results."""

# Document type label used in the user prompt
DOC_TYPE = "Product Description"


def build_context(group):
    """
    Takes all rows for one query_id, sorts by rank, re-numbers 1..N.
    Returns:
        docs: list of doc_text in retrieval order
        target_new_position: 0-based position of target doc after re-numbering
    """
    group = group.sort_values("rank").reset_index(drop=True)
    docs = group["doc_text"].tolist()

    # Find 0-based position of the target document
    target_new_position = None
    for i, row in group.iterrows():
        if row["is_target"] == 1:
            target_new_position = i  # 0-based
            break

    return docs, target_new_position


def build_user_prompt(query, docs):
    """
    Builds the user prompt for the LLM.
    docs: list of doc_text strings, numbered 1..N
    """
    sources = ""
    for i, doc_text in enumerate(docs, start=1):
        sources += f"[{i}] {DOC_TYPE}: {doc_text}\n\n"
    return f"Question: {query}\n\nSearch Results:\n{sources}"


def extract_citation_order(response_text):
    """
    Extracts citation indices in order of first appearance.
    0-based: [1] in text -> index 0, [2] -> index 1, etc.
    e.g. "...product [2] is great [1][2]..." -> [1, 0]
    """
    citations = re.findall(r"\[(\d+)\]", response_text)
    seen = []
    for c in citations:
        idx = int(c) - 1  # 0-based
        if idx not in seen:
            seen.append(idx)
    return seen


print("Helper functions defined.")

Helper functions defined.


## Parameters

In [4]:
# LLM model used for response generation
LLM_NAME = "gpt-4o-mini-2024-07-18"

# Notebook ID — A processes 0%–25% of all runs
NOTEBOOK_ID = "B"

# Output file for this notebook's results — separate from B, C, D to allow parallel execution
output_path = os.path.join(data_dir, "generation/4_generation_results_B_v3.parquet")

# Based on Vertex AI / Gemini default inference parameters:
# https://docs.cloud.google.com/vertex-ai/generative-ai/docs/model-reference/inference?hl=de
TEMPERATURE       = 1.0
TOP_P             = 0.95
SEED              = 42
FREQUENCY_PENALTY = 0.0
PRESENCE_PENALTY  = 0.0

# Assign this notebook's quarter of all runs sorted alphabetically by query_id
all_run_ids_full = sorted(runs_with_target)
total = len(all_run_ids_full)
start_idx = int(total * 25/100)
end_idx   = int(total * 50/100)
all_run_ids = all_run_ids_full[start_idx:end_idx]

# Load already completed runs to support safe resume after interruption
if os.path.exists(output_path):
    df_done = pd.read_parquet(output_path)
    done_keys = set(df_done["query_id"].tolist())
    already_done = len(done_keys)
else:
    done_keys = set()
    already_done = 0

llm = OpenAIHelper(LLM_NAME)

remaining = len(all_run_ids) - already_done
print(f"Notebook:      {NOTEBOOK_ID}")
print(f"LLM:           {LLM_NAME}")
print(f"Range:         25%–50% ({start_idx}–{end_idx} of {total})")
print(f"Runs in range: {len(all_run_ids)}")
print(f"Already done:  {already_done}")
print(f"Remaining:     {remaining}")
print(f"Estimated time: ~{remaining * 9.5 / 3600:.1f} hours")
print(f"Estimated cost: ~${remaining * 0.0003:.2f}")

Notebook:      B
LLM:           gpt-4o-mini-2024-07-18
Range:         25%–50% (2746–5492 of 10985)
Runs in range: 2746
Already done:  0
Remaining:     2746
Estimated time: ~7.2 hours
Estimated cost: ~$0.82


## Generation

For each run (query_id):
1. Sort docs by rank and re-number 1..N
2. Build LLM prompt
3. Generate response with inline citations
4. Extract citation order
5. Save incrementally

Runs without a target doc are skipped.
Safe to interrupt and resume.

In [5]:
# Load existing results if the output file already exists
# Allows safe resume without reprocessing completed runs
if os.path.exists(output_path):
    results_df = pd.read_parquet(output_path)
    results = results_df.to_dict("records")
else:
    results = []

start_time = datetime.now()
print(f"Started at: {start_time.strftime('%H:%M:%S')}")
print(f"Total runs to process: {len(all_run_ids)}")
print()

for i, query_id in enumerate(all_run_ids):

    # Skip runs already completed in a previous session
    if query_id in done_keys:
        continue

    # Load all documents for this run
    group = df[df["query_id"] == query_id]
    query_text         = group["query"].iloc[0]
    original_query_id  = group["original_query_id"].iloc[0]
    method             = group["method"].iloc[0]

    # Sort docs by retrieval rank and find target position
    docs, target_new_position = build_context(group)

    # Skip runs where the target document is not in the retrieval results
    if target_new_position is None:
        print(f"  [{i+1}] {query_id}: no target — skipping")
        continue

    # Build LLM prompt with numbered search results
    user_prompt = build_user_prompt(query_text, docs)
    messages = [
        {"role": "system", "content": DEVELOPER_PROMPT},
        {"role": "user",   "content": user_prompt},
    ]

    try:
        # Generate LLM response and extract citation order
        response, _ = llm.generate(messages)
        response_text    = response.content
        citation_order   = extract_citation_order(response_text)

        # Store result and save incrementally — no work lost on crash or interruption
        results.append({
            "query_id":            query_id,
            "original_query_id":   original_query_id,
            "method":              method,
            "query":               query_text,
            "target_new_position": target_new_position,
            "num_docs":            len(docs),
            "citation_order":      citation_order,
            "llm_response":        response_text,
        })
        done_keys.add(query_id)
        pd.DataFrame(results).to_parquet(output_path, index=False)
        print(f"  [{i+1}/{len(all_run_ids)}] {query_id}: target at [{target_new_position}] | cited: {citation_order[:5]}")

    except Exception as e:
        # First failure — wait 10 seconds and retry once (handles rate limits)
        print(f"  [{i+1}/{len(all_run_ids)}] {query_id}: ERROR — {e}")
        time.sleep(10)
        try:
            response, _    = llm.generate(messages)
            response_text  = response.content
            citation_order = extract_citation_order(response_text)
            results.append({
                "query_id":            query_id,
                "original_query_id":   original_query_id,
                "method":              method,
                "query":               query_text,
                "target_new_position": target_new_position,
                "num_docs":            len(docs),
                "citation_order":      citation_order,
                "llm_response":        response_text,
            })
            done_keys.add(query_id)
            pd.DataFrame(results).to_parquet(output_path, index=False)
            print(f"  [{i+1}/{len(all_run_ids)}] {query_id}: done (retry OK)")
        except Exception as e2:
            # Second failure — skip this run and continue with the next
            print(f"  [{i+1}/{len(all_run_ids)}] {query_id}: FAILED — {e2}")

# Summary
end_time = datetime.now()
elapsed  = end_time - start_time
print(f"{'='*60}")
print(f"GENERATION COMPLETE")
print(f"Started:    {start_time.strftime('%H:%M:%S')}")
print(f"Finished:   {end_time.strftime('%H:%M:%S')}")
print(f"Total time: {str(elapsed).split('.')[0]}")
print(f"Results:    {len(results)} rows saved to {output_path}")

Started at: 23:12:19
Total runs to process: 2746

  [1/2746] 18938_UniqueWords(doc): target at [8] | cited: [0, 1, 4, 5, 9]
  [2/2746] 18938_doc: target at [7] | cited: [0, 2, 5, 6, 9]
  [3/2746] 19044_Authoritative(doc): target at [1] | cited: [1, 2, 4, 0, 7]
  [4/2746] 19044_CQ(doc): target at [8] | cited: [0, 1, 5, 4, 6]
  [5/2746] 19044_CQS(doc): target at [8] | cited: [0, 1, 4, 3, 5]
  [6/2746] 19044_CS(doc): target at [8] | cited: [0, 1, 4, 3, 7]
  [7/2746] 19044_Citations(doc): target at [8] | cited: [0, 2, 8, 1, 5]
  [8/2746] 19044_ContentImprovement(doc): target at [9] | cited: [0, 1, 4, 3, 5]
  [9/2746] 19044_FC(doc): target at [9] | cited: [0, 4, 7, 6, 3]
  [10/2746] 19044_FCQ(doc): target at [7] | cited: [0, 2, 4, 8, 1]
  [11/2746] 19044_FCQS(doc): target at [1] | cited: [0, 1, 5, 3, 9]
  [12/2746] 19044_FCS(doc): target at [8] | cited: [0, 1, 4, 2, 3]
  [13/2746] 19044_FQ(doc): target at [9] | cited: [0, 1, 5, 8, 9]
  [14/2746] 19044_FQS(doc): target at [9] | cited: [0, 1,

## Inspect Results

In [6]:
# Load this notebook's parquet file for inspection
df_results = pd.read_parquet(output_path)

# Change INSPECT_QUERY_ID to inspect a specific run
INSPECT_QUERY_ID = df_results["query_id"].iloc[0]

row = df_results[df_results["query_id"] == INSPECT_QUERY_ID].iloc[0]

print(f"Run (query_id):    {row['query_id']}")
print(f"Original query:    {row['original_query_id']}")
print(f"Method:            {row['method']}")
print(f"Query:             {row['query']}")
print(f"Num docs:          {row['num_docs']}")
print(f"Target position:   {row['target_new_position']} (0-based)")
print(f"Citation order:    {row['citation_order']}")
print(f"Target cited:      {'YES' if row['target_new_position'] in row['citation_order'] else 'NO'}")
print(f"\n--- LLM Response ---")
print(row["llm_response"][:800])

Run (query_id):    18938_UniqueWords(doc)
Original query:    18938
Method:            UniqueWords(doc)
Query:             bouncy house for kids outdoor large
Num docs:          10
Target position:   8 (0-based)
Citation order:    [0 1 4 5 9]
Target cited:      NO

--- LLM Response ---
For an engaging outdoor play experience, consider several options for large inflatable bounce houses suitable for kids. Here are a few notable products:

1. **OTTARO Bounce House**: This inflatable bouncy house is made from heavy-duty, puncture-proof materials (420D Oxford) and features a large jumping area reinforced with extra thick fabric (840D Oxford) to enhance durability. It is designed for up to three children at a time, with a total weight limit of 220 lbs. The setup is quick, inflating in under two minutes, and includes protective netting to ensure safety during play, making it an excellent choice for backyard fun [1].

2. **BOUNTECH Inflatable Bounce House**: With a dimension of 9.8ft by 9.8ft a

## Merge Results

**Run only after all 4 notebooks (A, B, C, D) are complete.**

In [ ]:
# Merge all four notebook parquet files into one combined file
# Run only after notebooks A, B, C, D have all completed
NOTEBOOK_IDS = ["A", "B", "C", "D"]
dfs = []
for nb in NOTEBOOK_IDS:
    path = os.path.join(data_dir, f"generation/4_generation_results_{nb}_v3.parquet")
    if os.path.exists(path):
        dfs.append(pd.read_parquet(path))
        print(f"Loaded 4_generation_results_{nb}_v3.parquet ({len(dfs[-1])} rows)")
    else:
        print(f"WARNING: 4_generation_results_{nb}_v3.parquet not found — skipping")

# Concatenate all four subsets into one merged DataFrame
merged      = pd.concat(dfs, ignore_index=True)
merged_path = os.path.join(data_dir, "generation/4_generation_results_v3.parquet")
merged.to_parquet(merged_path, index=False)
print(f"\nMerged {len(merged)} rows into 4_generation_results_v3.parquet")

## Summary

In [ ]:
# Load merged parquet and verify completeness
# Expected: total rows == total runs with target across all 4 notebooks
merged_path = os.path.join(data_dir, "generation/4_generation_results_v3.parquet")
df_results  = pd.read_parquet(merged_path)

print(f"Total runs done:     {len(df_results)}")
print(f"Total runs expected: {len(all_run_ids_full)}")
print(f"Missing runs:        {len(all_run_ids_full) - len(df_results)}")
print()
print(f"Unique methods: {df_results['method'].nunique()}")
print(f"Unique queries: {df_results['original_query_id'].nunique()}")